In [ ]:
# importing required libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, Docx2txtLoader
import os
from docx import Document
from langchain_astradb import AstraDBVectorStore

c:\Users\Namshima\anaconda3\envs\brightpath\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_DIR = "./data"

In [ ]:
def load_docx(filepath: str) -> str:
    """
    Extracts and returns cleaned text from a .docx file.

    Args:
        filepath (str): Path to the .docx file.

    Returns:
        str: A single string containing all non-empty paragraphs,
        joined by newline characters.
    """
    doc = Document(filepath)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n".join(paragraphs)

In [ ]:
def load_all_documents(data_dir: str) -> list[dict]:
     """
    Loads and extracts text from all valid .docx files in a directory.

    - Skips temporary or invalid files (e.g., files starting with '~$')
    - Extracts text using `load_docx` and compiles metadata for each document
    Args:
        data_dir (str): Path to the directory containing .docx files.

    Returns:
        list[dict]: A list of dictionaries where each item contains:
            - "filename" (str): Name of the document
            - "text" (str): Extracted and cleaned full text from the document
    """
     documents = []
     for filename in sorted(os.listdir(data_dir)):
        if filename.startswith("~$"):
            continue # Skip temporary files created by Word

        if not filename.endswith(".docx"):
            continue

        filepath = os.path.join(data_dir, filename)
        text = load_docx(filepath)

        documents.append({
            "filename": filename,
            "text": text
        })

        print(f"Loaded: {filename}")
        print(f"  Characters : {len(text)}")
        print(f"  Words      : {len(text.split())}")
        print(f"  Preview    : {text[:120]}...")
        print()

        return documents

docs = load_all_documents(DATA_DIR)

Loaded: 01_BrightPath_FAQs.docx
  Characters : 8467
  Words      : 1383
  Preview    : This document addresses the most common questions received by BrightPath Academy's student support team across admission...

Loaded: 02_BrightPath_Course_Catalogue.docx
  Characters : 427
  Words      : 58
  Preview    : BrightPath Academy offers over 40 programmes across five learning tracks: Technology & Data, Business & Management, Crea...

Loaded: 03_BrightPath_Student_Policies.docx
  Characters : 5776
  Words      : 841
  Preview    : This handbook sets out BrightPath Academy's official policies governing learner rights and responsibilities. All enrolle...

Loaded: 04_BrightPath_Onboarding_Guide.docx
  Characters : 3981
  Words      : 625
  Preview    : Congratulations on enrolling at BrightPath Academy! This guide will walk you through everything you need to know to get ...

Loaded: 05_BrightPath_Contact_Support.docx
  Characters : 1075
  Words      : 145
  Preview    : This directory lists all

In [5]:
loader = DirectoryLoader(
    "./data",
    glob="**/*.docx", 
    loader_cls=Docx2txtLoader,    # Extracts text/tables from DOCX
    silent_errors=True
    )

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)


In [7]:
CHUNK_THRESHOLD = 450  # Files smaller than this won't be split
final_documents = []


for filename in os.listdir(DATA_DIR):
    
    if filename.startswith("~$"):
        continue # Skip temporary files created by Word

    if filename.endswith(".docx"):
        loader = Docx2txtLoader(os.path.join(DATA_DIR, filename))
        data = loader.load()
        
        # Check total character count for the file
        doc_content = data[0].page_content
        char_count = len(doc_content)
        
        if char_count < CHUNK_THRESHOLD:
            # Add as a single 'Non-Chunkable' unit
            print(f"Keeping {filename} whole ({char_count} chars)")
            final_documents.extend(data)
        else:
            # Perform standard chunking
            print(f"Chunking {filename} ({char_count} chars)")
            chunks = text_splitter.split_documents(data)
            final_documents.extend(chunks)

Chunking 01_BrightPath_FAQs.docx (8980 chars)
Chunking 02_BrightPath_Course_Catalogue.docx (7922 chars)
Chunking 03_BrightPath_Student_Policies.docx (7465 chars)
Chunking 04_BrightPath_Onboarding_Guide.docx (5889 chars)
Chunking 05_BrightPath_Contact_Support.docx (5477 chars)


In [ ]:
ASTRA_DB_API_ENDPOINT=''
ASTRA_DB_APPLICATION_TOKEN=''

In [10]:
# Embeddings
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 666.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Initialize AstraDB
vstore = AstraDBVectorStore(
    collection_name="",
    embedding=embeddings,
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT
)

# Add documents to vector database
vstore.add_documents(final_documents)


In [ ]:
from astrapy import DataAPIClient

client = DataAPIClient(ASTRA_DB_APPLICATION_TOKEN)
db = client.get_database_by_api_endpoint(ASTRA_DB_API_ENDPOINT)

collections = db.list_collections()

print(collections)

[CollectionDescriptor(name='brightpath_rag', definition=CollectionDefinition(vector=CollectionVectorOptions(dimension=384, metric='cosine', source_model='other', service=None), lexical=CollectionLexicalOptions(enabled=False, analyzer=None), rerank=CollectionRerankOptions(enabled=False, service=None), indexing={'allow': ['metadata']}), raw_descriptor=...)]


In [11]:
vector_1 = embeddings.embed_query(final_documents[0].page_content)
vector_2 = embeddings.embed_query(final_documents[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 384

[0.013517308048903942, -0.08209134638309479, 0.029090408235788345, 0.0815201923251152, 0.018379202112555504, 0.001332130515947938, -0.031083086505532265, 0.005812872666865587, -0.13998696208000183, -0.018452849239110947]


In [20]:
results = vstore.similarity_search(
    "what courses do you offer"
)

print(results[0])

page_content='Q: How do I create an account and enrol in a course?' metadata={'source': './data\\01_BrightPath_FAQs.docx'}
